# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samarthjoshi02/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

We load the raw dataset and engineer features. For missing values, we fill numeric values with `0` (noting that a blind fillna injects category signals due to missingness mapping to content types) and categoricals with `'unknown'`. We construct our label `is_declining_label` from `trend_direction`.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Load data
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Build Target
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Numeric Fills
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)

# Categorical Fills
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
for col in cat_cols:
    df[col] = df[col].fillna('unknown').astype(str)

# Show base rate
print(f"Total Rows: {len(df)}")
print(f"Base Rate (is_declining_label = 1): {df['is_declining_label'].mean():.2%}")


Total Rows: 30000
Base Rate (is_declining_label = 1): 54.21%


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

1. **`search_volume`**: Keyword volume. Missing values (e.g., feedly articles) are filled with 0. Exists prior to prediction.
2. **`impressions_90d`**: GSC trailing 90-day impressions. Known strictly prior. Missing filled with 0.
3. **`ai_traffic_pct`**: AI referred sessions %. Can exceed 100%. Known prior.
4. **`impressions_last_30d`**: **DANGER**. This overlaps with our label definition (last 30 days vs prev 30 days). Not available *before* the moment we predict (if predicting the last 30 days).


In [2]:
# We can inspect the overlap between our feature window and the target window
# The label is derived from `trend_direction` which uses `impressions_last_30d` and `impressions_prev_30d`.
print("Features that are unsafe (overlap with label):")
print(["impressions_last_30d", "clicks_last_30d", "sessions_last_30d", "trend_pct", "trend_direction"])


Features that are unsafe (overlap with label):
['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'trend_pct', 'trend_direction']


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

We will train a model with the leaky `trend_pct` feature and one without it. A perfect score (AUC ~ 1.0) is a confession of leakage.


In [3]:
# Define leaky features and safe features
leaky_features = ['trend_pct', 'impressions_last_30d', 'impressions_prev_30d']
safe_features = ['impressions_90d', 'clicks_90d', 'search_volume', 'word_count']

# Grouped Split to be honest
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(df, groups=df['client_id']))

train, test = df.iloc[train_idx], df.iloc[test_idx]

# Model WITH leakage
rf_leaky = RandomForestClassifier(random_state=42, n_estimators=50)
rf_leaky.fit(train[leaky_features], train['is_declining_label'])
preds_leaky = rf_leaky.predict_proba(test[leaky_features])[:, 1]
auc_leaky = roc_auc_score(test['is_declining_label'], preds_leaky)

# Model WITHOUT leakage (Honest)
rf_safe = RandomForestClassifier(random_state=42, n_estimators=50)
rf_safe.fit(train[safe_features], train['is_declining_label'])
preds_safe = rf_safe.predict_proba(test[safe_features])[:, 1]
auc_safe = roc_auc_score(test['is_declining_label'], preds_safe)

print(f"AUC WITH Leaky Features (`trend_pct`): {auc_leaky:.4f} (Too good to be true!)")
print(f"AUC WITHOUT Leaky Features (Safe): {auc_safe:.4f}")


AUC WITH Leaky Features (`trend_pct`): 1.0000 (Too good to be true!)
AUC WITHOUT Leaky Features (Safe): 0.5612


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- **`trend_direction`**: Directly computes the target label (`is_declining_label = trend_direction == 'down'`).
- **`trend_pct`**: Derived from the exact same future window as the label; dictates `trend_direction`.
- **`impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`**: These represent the future outcome window. They are not knowable at the moment of prediction.
- **`content_id`, `client_id`**: Pseudonymized identifiers that do not generalize; used only for grouped splitting.
- **`provider_used`, `model_used`**: Specifically stated in the dictionary as "Not a model feature".


In [4]:
# Final list of safely usable features
print("We strictly exclude all overlapping time windows and label-derived columns to maintain an honest evaluation.")


We strictly exclude all overlapping time windows and label-derived columns to maintain an honest evaluation.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
